In [0]:
energy_df = spark.read.csv(
    "/Volumes/pysparkazure_7405617424734388/default/week4/energy_usage.csv", header=True, inferSchema=True
)

In [0]:
sensor_df = spark.read.csv(
    "/Volumes/pysparkazure_7405617424734388/default/week4/sensor_logs.csv", header=True, inferSchema=True
)


In [0]:
display(sensor_df)

device_id,device_name,timestamp,hour,energy_kwh
1,Air Conditioner,2026-01-01T14:00:00.000Z,14,2.5
1,Air Conditioner,2026-01-01T20:00:00.000Z,20,2.3
1,Air Conditioner,2026-01-02T15:00:00.000Z,15,2.7
2,Refrigerator,2026-01-01T00:00:00.000Z,0,1.1
2,Refrigerator,2026-01-01T12:00:00.000Z,12,1.05
2,Refrigerator,2026-01-02T01:00:00.000Z,1,1.15
3,Washing Machine,2026-01-01T09:00:00.000Z,9,0.9
3,Washing Machine,2026-01-02T10:00:00.000Z,10,1.0
4,Television,2026-01-01T19:00:00.000Z,19,0.3
4,Television,2026-01-02T21:00:00.000Z,21,0.25


In [0]:
display(energy_df)

timestamp,device_id,room_id,energy_kwh
2026-01-01T14:00:00.000Z,1,1,2.5
2026-01-01T20:00:00.000Z,1,1,2.3
2026-01-01T00:00:00.000Z,2,2,1.1
2026-01-01T12:00:00.000Z,2,2,1.05
2026-01-01T09:00:00.000Z,3,3,0.9
2026-01-01T19:00:00.000Z,4,1,0.3
2026-01-01T07:00:00.000Z,5,4,3.2
2026-01-02T15:00:00.000Z,1,1,2.7
2026-01-02T01:00:00.000Z,2,2,1.15
2026-01-02T10:00:00.000Z,3,3,null


In [0]:
from pyspark.sql import functions as F

sensor_df = sensor_df.withColumn("log_date", F.to_date("timestamp"))

In [0]:
daily_summary = sensor_df.groupBy("device_id", "device_name", "log_date") \
    .agg(F.sum("energy_kwh").alias("daily_kwh"))

weekly_summary = sensor_df.groupBy("device_id", "device_name") \
    .agg(F.sum("energy_kwh").alias("weekly_kwh"))

display(daily_summary)
display(weekly_summary)

device_id,device_name,log_date,daily_kwh
5,Water Heater,2026-01-02,3.1
4,Television,2026-01-01,0.3
1,Air Conditioner,2026-01-02,2.7
5,Water Heater,2026-01-01,3.2
1,Air Conditioner,2026-01-01,4.8
5,Water Heater,2026-01-03,6.3
3,Washing Machine,2026-01-01,0.9
4,Television,2026-01-02,0.25
4,Television,2026-01-03,0.35
2,Refrigerator,2026-01-01,2.1500000000000004


device_id,device_name,weekly_kwh
3,Washing Machine,1.9
5,Water Heater,12.600000000000001
1,Air Conditioner,7.5
4,Television,0.9
2,Refrigerator,3.3000000000000003


In [0]:
flagged = daily_summary.withColumn(
    "alert",
    F.when(F.col("daily_kwh") > 10, "HIGH USAGE").otherwise("OK")
)

display(flagged)

device_id,device_name,log_date,daily_kwh,alert
5,Water Heater,2026-01-02,3.1,OK
4,Television,2026-01-01,0.3,OK
1,Air Conditioner,2026-01-02,2.7,OK
5,Water Heater,2026-01-01,3.2,OK
1,Air Conditioner,2026-01-01,4.8,OK
5,Water Heater,2026-01-03,6.3,OK
3,Washing Machine,2026-01-01,0.9,OK
4,Television,2026-01-02,0.25,OK
4,Television,2026-01-03,0.35,OK
2,Refrigerator,2026-01-01,2.1500000000000004,OK


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS energy_tracker")

flagged.write.format("delta").mode("overwrite").saveAsTable("energy_tracker.daily_summary")
daily_summary.toPandas().to_csv(
    "daily_summary.csv",
    index=False
)

weekly_summary.toPandas().to_csv(
    "weekly_summary.csv",
    index=False
)

flagged.toPandas().to_csv(
    "energy_alerts.csv",
    index=False
)
print("Saved as Delta table 'energy_tracker.daily_summary' and as CSV.")

Saved as Delta table 'energy_tracker.daily_summary' and as CSV.


In [0]:
flagged = daily_summary.withColumn(
    "alert",
    F.when(F.col("daily_kwh") > 5, "HIGH USAGE")
     .otherwise("OK")
)

In [0]:
flagged.createOrReplaceTempView("daily_summary")
weekly_summary.createOrReplaceTempView("weekly_summary_view")

In [0]:

spark.sql("""
SELECT
    device_name,
    weekly_kwh
FROM weekly_summary_view
WHERE weekly_kwh > (
    SELECT AVG(weekly_kwh)
    FROM weekly_summary_view
)
ORDER BY weekly_kwh DESC
""").show(truncate=False)

+---------------+------------------+
|device_name    |weekly_kwh        |
+---------------+------------------+
|Water Heater   |12.600000000000001|
|Air Conditioner|7.5               |
+---------------+------------------+



In [0]:
spark.sql("""
SELECT device_name,
       ROUND(weekly_kwh, 2) AS weekly_usage
FROM weekly_summary_view
ORDER BY weekly_kwh DESC
LIMIT 1
""").show(truncate=False)

+------------+------------+
|device_name |weekly_usage|
+------------+------------+
|Water Heater|12.6        |
+------------+------------+

